In [2]:
from dotenv import load_dotenv
import os

load_dotenv(override=True)

print("API loaded:", bool(os.getenv("GEMINI_API_KEY")))

API loaded: True


In [1]:
# ================================================================
# MAIZE SEED RAG - 20 QUERY PIPELINE EVALUATION
# ================================================================
#
# Run this entire cell in your Jupyter Notebook.
#
# It evaluates:
#   1. Metadata retrieval
#   2. Region matching
#   3. Season matching
#   4. Maturity constraint
#   5. Irrigation handling
#   6. Gemini recommendation
#   7. Recommendation validity
#   8. Candidate-grounding / hallucination
#   9. Overall pipeline success
#
# Output:
#   - Detailed query-level results
#   - Aggregate metrics
#   - CSV report
#   - JSON report
#
# ================================================================


import sys
import os
import json
import re
import time
from pathlib import Path
from datetime import datetime

import pandas as pd


# ================================================================
# 1. PROJECT PATH
# ================================================================

PROJECT_ROOT = Path.cwd()

# If notebook is inside:
# maize-seed-rag/notebooks/
# move one level up.

if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("=" * 80)
print("MAIZE SEED RAG - PIPELINE EVALUATION")
print("=" * 80)
print()
print("Project root:")
print(PROJECT_ROOT)
print()


# ================================================================
# 2. IMPORT YOUR EXISTING PIPELINE
# ================================================================

from src.retrieval.metadata_filter import (
    load_documents,
    filter_varieties
)

from gemini_recommender import recommend_varieties


# ================================================================
# 3. LOAD DOCUMENTS
# ================================================================

documents = load_documents()

print("Total documents loaded:", len(documents))
print()


# ================================================================
# 4. HELPER FUNCTIONS
# ================================================================

def normalize_text(value):
    """
    Normalize text for comparison.
    """
    if value is None:
        return ""

    return str(value).strip().lower()


def get_metadata(document):
    return document.get("metadata", {})


def get_variety_id(document):
    metadata = get_metadata(document)

    return (
        metadata.get("variety_id")
        or document.get("id")
        or ""
    )


def get_variety_name(document):
    metadata = get_metadata(document)

    return str(
        metadata.get("variety_name", "")
    )


def safe_float(value):
    try:
        return float(value)
    except:
        return None


def maturity_matches(document, max_days=None, min_days=None):
    """
    Checks whether a candidate satisfies the requested
    maturity constraints.
    """

    metadata = get_metadata(document)

    min_maturity = safe_float(
        metadata.get("maturity_days_min")
    )

    max_maturity = safe_float(
        metadata.get("maturity_days_max")
    )

    if min_maturity is None or max_maturity is None:
        return False

    if max_days is not None:
        if max_maturity > max_days:
            return False

    if min_days is not None:
        if min_maturity < min_days:
            return False

    return True


def region_matches(document, state):
    """
    Checks whether requested state appears in
    recommended regions.
    """

    if not state:
        return True

    metadata = get_metadata(document)

    regions = normalize_text(
        metadata.get("recommended_regions")
    )

    return normalize_text(state) in regions


def season_matches(document, season):
    """
    Checks whether requested season appears
    in candidate metadata.
    """

    if not season:
        return True

    metadata = get_metadata(document)

    candidate_season = normalize_text(
        metadata.get("season")
    )

    return normalize_text(season) in candidate_season


def irrigation_value(document):
    metadata = get_metadata(document)

    return metadata.get(
        "irrigation_condition"
    )


def irrigation_match_type(document):
    metadata = get_metadata(document)

    return metadata.get(
        "irrigation_match_type"
    )


# ================================================================
# 5. VALIDATE GEMINI RECOMMENDATION
# ================================================================

def validate_gemini_result(
    recommendation,
    retrieved_candidates
):
    """
    Evaluates whether Gemini recommendations are grounded
    in the retrieved candidate set.
    """

    result = {
        "valid_structure": False,
        "recommendation_count": 0,
        "all_candidates_grounded": True,
        "invalid_candidate_count": 0,
        "ranks_valid": True,
        "maturity_valid": True,
        "irrigation_handling_valid": True,
        "warnings_present": False
    }

    if not isinstance(
        recommendation,
        dict
    ):
        return result

    recommendations = recommendation.get(
        "recommendations"
    )

    if not isinstance(
        recommendations,
        list
    ):
        return result

    result["valid_structure"] = True

    result["recommendation_count"] = len(
        recommendations
    )

    if recommendation.get("warnings"):
        result["warnings_present"] = True

    retrieved_ids = set()

    for document in retrieved_candidates:

        metadata = get_metadata(document)

        variety_id = (
            metadata.get("variety_id")
            or document.get("id")
        )

        if variety_id:
            retrieved_ids.add(
                normalize_text(variety_id)
            )

    ranks = []

    for recommendation_item in recommendations:

        if not isinstance(
            recommendation_item,
            dict
        ):
            result["all_candidates_grounded"] = False
            result["invalid_candidate_count"] += 1
            continue

        variety_id = normalize_text(
            recommendation_item.get(
                "variety_id"
            )
        )

        # --------------------------------------------------------
        # Candidate grounding
        # --------------------------------------------------------

        if variety_id not in retrieved_ids:

            result["all_candidates_grounded"] = False
            result["invalid_candidate_count"] += 1

        # --------------------------------------------------------
        # Rank validation
        # --------------------------------------------------------

        rank = recommendation_item.get(
            "rank"
        )

        if isinstance(rank, int):
            ranks.append(rank)
        else:
            result["ranks_valid"] = False

        # --------------------------------------------------------
        # Maturity validation
        # --------------------------------------------------------

        maturity = str(
            recommendation_item.get(
                "maturity",
                ""
            )
        )

        if not maturity.strip():
            result["maturity_valid"] = False

        # --------------------------------------------------------
        # Irrigation validation
        # --------------------------------------------------------

        irrigation = str(
            recommendation_item.get(
                "irrigation",
                ""
            )
        )

        if not irrigation.strip():
            result["irrigation_handling_valid"] = False

    # ------------------------------------------------------------
    # Check ranks
    # ------------------------------------------------------------

    if ranks:

        expected_ranks = list(
            range(
                1,
                len(ranks) + 1
            )
        )

        if sorted(ranks) != expected_ranks:
            result["ranks_valid"] = False

    return result


# ================================================================
# 6. DEFINE 20 EVALUATION QUERIES
# ================================================================
#
# These represent different combinations of:
#
#   State
#   Season
#   Irrigation
#   Maturity
#
# The expected constraints are evaluated against the
# retrieved documents.
#
# ================================================================

TEST_QUERIES = [

    {
        "query_id": "Q01",
        "query": (
            "I am a farmer from Andhra Pradesh. "
            "I want to grow maize during Kharif season "
            "under rainfed conditions and I need a variety "
            "that matures within 100 days."
        ),
        "state": "Andhra Pradesh",
        "season": "Kharif",
        "irrigation": "rainfed",
        "max_maturity_days": 100,
        "min_maturity_days": None
    },

    {
        "query_id": "Q02",
        "query": (
            "I am from Telangana and want to grow maize "
            "during Kharif. I need a variety that matures "
            "within 100 days under rainfed conditions."
        ),
        "state": "Telangana",
        "season": "Kharif",
        "irrigation": "rainfed",
        "max_maturity_days": 100,
        "min_maturity_days": None
    },

    {
        "query_id": "Q03",
        "query": (
            "I am a farmer in Karnataka. "
            "I want a Kharif maize variety with maturity "
            "below 100 days."
        ),
        "state": "Karnataka",
        "season": "Kharif",
        "irrigation": None,
        "max_maturity_days": 100,
        "min_maturity_days": None
    },

    {
        "query_id": "Q04",
        "query": (
            "I am farming maize in Maharashtra during Kharif "
            "and need a variety that matures within 90 days."
        ),
        "state": "Maharashtra",
        "season": "Kharif",
        "irrigation": None,
        "max_maturity_days": 90,
        "min_maturity_days": None
    },

    {
        "query_id": "Q05",
        "query": (
            "I am from Tamil Nadu and want to grow maize in "
            "Kharif season. The variety should mature within "
            "100 days."
        ),
        "state": "Tamil Nadu",
        "season": "Kharif",
        "irrigation": None,
        "max_maturity_days": 100,
        "min_maturity_days": None
    },

    {
        "query_id": "Q06",
        "query": (
            "I am a farmer from Rajasthan. "
            "I want a Kharif maize variety that matures "
            "within 100 days."
        ),
        "state": "Rajasthan",
        "season": "Kharif",
        "irrigation": None,
        "max_maturity_days": 100,
        "min_maturity_days": None
    },

    {
        "query_id": "Q07",
        "query": (
            "I am from Gujarat and want a maize variety "
            "for Kharif season with maturity below 100 days."
        ),
        "state": "Gujarat",
        "season": "Kharif",
        "irrigation": None,
        "max_maturity_days": 100,
        "min_maturity_days": None
    },

    {
        "query_id": "Q08",
        "query": (
            "I want to grow maize in Madhya Pradesh during "
            "Kharif season. Recommend varieties maturing "
            "within 100 days."
        ),
        "state": "Madhya Pradesh",
        "season": "Kharif",
        "irrigation": None,
        "max_maturity_days": 100,
        "min_maturity_days": None
    },

    {
        "query_id": "Q09",
        "query": (
            "I am a maize farmer in Chhattisgarh. "
            "I need a Kharif variety that matures within "
            "100 days."
        ),
        "state": "Chhattisgarh",
        "season": "Kharif",
        "irrigation": None,
        "max_maturity_days": 100,
        "min_maturity_days": None
    },

    {
        "query_id": "Q10",
        "query": (
            "I am from Andhra Pradesh and need a maize "
            "variety for Kharif season that matures between "
            "80 and 100 days."
        ),
        "state": "Andhra Pradesh",
        "season": "Kharif",
        "irrigation": None,
        "max_maturity_days": 100,
        "min_maturity_days": 80
    },

    {
        "query_id": "Q11",
        "query": (
            "I am from Andhra Pradesh. I want to cultivate "
            "maize during Kharif under rainfed conditions "
            "and prefer maturity below 90 days."
        ),
        "state": "Andhra Pradesh",
        "season": "Kharif",
        "irrigation": "rainfed",
        "max_maturity_days": 90,
        "min_maturity_days": None
    },

    {
        "query_id": "Q12",
        "query": (
            "I am from Telangana and need a maize variety "
            "for Kharif that matures between 80 and 100 days."
        ),
        "state": "Telangana",
        "season": "Kharif",
        "irrigation": None,
        "max_maturity_days": 100,
        "min_maturity_days": 80
    },

    {
        "query_id": "Q13",
        "query": (
            "I am from Karnataka and want a Kharif maize "
            "variety with maturity between 50 and 90 days."
        ),
        "state": "Karnataka",
        "season": "Kharif",
        "irrigation": None,
        "max_maturity_days": 90,
        "min_maturity_days": 50
    },

    {
        "query_id": "Q14",
        "query": (
            "I am from Maharashtra and want a Kharif maize "
            "variety that matures within 100 days under "
            "rainfed farming."
        ),
        "state": "Maharashtra",
        "season": "Kharif",
        "irrigation": "rainfed",
        "max_maturity_days": 100,
        "min_maturity_days": None
    },

    {
        "query_id": "Q15",
        "query": (
            "I am a farmer in Andhra Pradesh. "
            "Recommend a Kharif maize variety with maturity "
            "below 60 days."
        ),
        "state": "Andhra Pradesh",
        "season": "Kharif",
        "irrigation": None,
        "max_maturity_days": 60,
        "min_maturity_days": None
    },

    {
        "query_id": "Q16",
        "query": (
            "I am from Telangana and need a Kharif maize "
            "variety maturing below 70 days."
        ),
        "state": "Telangana",
        "season": "Kharif",
        "irrigation": None,
        "max_maturity_days": 70,
        "min_maturity_days": None
    },

    {
        "query_id": "Q17",
        "query": (
            "I am from Andhra Pradesh and want maize during "
            "Kharif. I need a variety that matures within "
            "85 days."
        ),
        "state": "Andhra Pradesh",
        "season": "Kharif",
        "irrigation": None,
        "max_maturity_days": 85,
        "min_maturity_days": None
    },

    {
        "query_id": "Q18",
        "query": (
            "I am from Tamil Nadu and need a Kharif maize "
            "variety maturing within 90 days under rainfed "
            "conditions."
        ),
        "state": "Tamil Nadu",
        "season": "Kharif",
        "irrigation": "rainfed",
        "max_maturity_days": 90,
        "min_maturity_days": None
    },

    {
        "query_id": "Q19",
        "query": (
            "I am from Andhra Pradesh and want to grow maize "
            "in Kharif season. I need a variety that matures "
            "within 100 days and is suitable for my region."
        ),
        "state": "Andhra Pradesh",
        "season": "Kharif",
        "irrigation": None,
        "max_maturity_days": 100,
        "min_maturity_days": None
    },

    {
        "query_id": "Q20",
        "query": (
            "I am from Karnataka and want to grow maize during "
            "Kharif. I need a variety that matures within "
            "100 days and preferably works under rainfed "
            "conditions."
        ),
        "state": "Karnataka",
        "season": "Kharif",
        "irrigation": "rainfed",
        "max_maturity_days": 100,
        "min_maturity_days": None
    }
]


# ================================================================
# 7. EVALUATE ONE QUERY
# ================================================================

def evaluate_query(test_case, query_number):

    print()
    print("=" * 80)
    print(
        f"EVALUATING QUERY {query_number}/"
        f"{len(TEST_QUERIES)}"
    )
    print("=" * 80)

    print()
    print(test_case["query"])

    state = test_case["state"]
    season = test_case["season"]
    irrigation = test_case["irrigation"]
    max_maturity_days = test_case[
        "max_maturity_days"
    ]
    min_maturity_days = test_case[
        "min_maturity_days"
    ]

    result_row = {
        "query_id": test_case["query_id"],
        "query": test_case["query"],
        "state": state,
        "season": season,
        "irrigation": irrigation,
        "max_maturity_days": max_maturity_days,
        "min_maturity_days": min_maturity_days,

        "retrieved_candidates": 0,
        "region_valid_candidates": 0,
        "season_valid_candidates": 0,
        "maturity_valid_candidates": 0,

        "metadata_constraint_accuracy": 0.0,

        "gemini_success": False,
        "gemini_model": None,

        "recommendation_count": 0,
        "recommendation_structure_valid": False,
        "recommendations_grounded": False,
        "invalid_recommendations": 0,
        "rank_valid": False,
        "recommendation_maturity_valid": False,
        "irrigation_handling_valid": False,

        "pipeline_success": False,

        "error": ""
    }

    # ============================================================
    # METADATA FILTERING
    # ============================================================

    start_time = time.time()

    try:

        candidates = filter_varieties(
            documents,
            state=state,
            season=season,
            irrigation=irrigation,
            max_maturity_days=max_maturity_days,
            min_maturity_days=min_maturity_days
        )

        result_row[
            "retrieved_candidates"
        ] = len(candidates)

        # --------------------------------------------------------
        # Evaluate metadata constraints
        # --------------------------------------------------------

        if candidates:

            region_valid = sum(
                region_matches(
                    document,
                    state
                )
                for document in candidates
            )

            season_valid = sum(
                season_matches(
                    document,
                    season
                )
                for document in candidates
            )

            maturity_valid = sum(
                maturity_matches(
                    document,
                    max_days=max_maturity_days,
                    min_days=min_maturity_days
                )
                for document in candidates
            )

            result_row[
                "region_valid_candidates"
            ] = region_valid

            result_row[
                "season_valid_candidates"
            ] = season_valid

            result_row[
                "maturity_valid_candidates"
            ] = maturity_valid

            total_checks = (
                len(candidates) * 3
            )

            passed_checks = (
                region_valid
                + season_valid
                + maturity_valid
            )

            if total_checks > 0:

                result_row[
                    "metadata_constraint_accuracy"
                ] = round(
                    passed_checks /
                    total_checks,
                    4
                )

        # ========================================================
        # GEMINI
        # ========================================================

        if candidates:

            try:

                recommendation = (
                    recommend_varieties(
                        test_case["query"],
                        candidates
                    )
                )

                # ------------------------------------------------
                # Your function returns a dictionary.
                # Some older versions may return a tuple.
                # Handle both safely.
                # ------------------------------------------------

                if isinstance(
                    recommendation,
                    tuple
                ):

                    if len(recommendation) >= 2:

                        recommendation = (
                            recommendation[0]
                        )

                # ------------------------------------------------
                # Model
                # ------------------------------------------------

                if isinstance(
                    recommendation,
                    dict
                ):

                    result_row[
                        "gemini_model"
                    ] = recommendation.get(
                        "model_used"
                    )

                    validation = (
                        validate_gemini_result(
                            recommendation,
                            candidates
                        )
                    )

                    result_row[
                        "gemini_success"
                    ] = True

                    result_row[
                        "recommendation_count"
                    ] = validation[
                        "recommendation_count"
                    ]

                    result_row[
                        "recommendation_structure_valid"
                    ] = validation[
                        "valid_structure"
                    ]

                    result_row[
                        "recommendations_grounded"
                    ] = validation[
                        "all_candidates_grounded"
                    ]

                    result_row[
                        "invalid_recommendations"
                    ] = validation[
                        "invalid_candidate_count"
                    ]

                    result_row[
                        "rank_valid"
                    ] = validation[
                        "ranks_valid"
                    ]

                    result_row[
                        "recommendation_maturity_valid"
                    ] = validation[
                        "maturity_valid"
                    ]

                    result_row[
                        "irrigation_handling_valid"
                    ] = validation[
                        "irrigation_handling_valid"
                    ]

            except Exception as e:

                result_row[
                    "error"
                ] = str(e)

        else:

            # No candidates is a valid retrieval outcome.
            result_row[
                "gemini_success"
            ] = False

    except Exception as e:

        result_row[
            "error"
        ] = str(e)

    # ============================================================
    # PIPELINE SUCCESS
    # ============================================================

    metadata_ok = (
        result_row[
            "metadata_constraint_accuracy"
        ] == 1.0
    )

    gemini_ok = (
        result_row[
            "gemini_success"
        ]
        and
        result_row[
            "recommendation_structure_valid"
        ]
        and
        result_row[
            "recommendations_grounded"
        ]
        and
        result_row[
            "rank_valid"
        ]
        and
        result_row[
            "recommendation_maturity_valid"
        ]
        and
        result_row[
            "irrigation_handling_valid"
        ]
    )

    result_row[
        "pipeline_success"
    ] = (
        metadata_ok
        and gemini_ok
    )

    elapsed = time.time() - start_time

    result_row[
        "execution_time_seconds"
    ] = round(
        elapsed,
        2
    )

    # ============================================================
    # PRINT RESULT
    # ============================================================

    print()
    print(
        "Retrieved candidates:",
        result_row["retrieved_candidates"]
    )

    print(
        "Metadata constraint accuracy:",
        f"{result_row['metadata_constraint_accuracy']:.2%}"
    )

    print(
        "Gemini success:",
        result_row["gemini_success"]
    )

    print(
        "Grounded recommendations:",
        result_row["recommendations_grounded"]
    )

    print(
        "Pipeline success:",
        result_row["pipeline_success"]
    )

    if result_row["error"]:

        print(
            "ERROR:",
            result_row["error"]
        )

    return result_row


# ================================================================
# 8. RUN ALL 20 TEST QUERIES
# ================================================================

evaluation_results = []

print()
print("=" * 80)
print("STARTING 20-QUERY EVALUATION")
print("=" * 80)
print()

for index, test_case in enumerate(
    TEST_QUERIES,
    start=1
):

    result = evaluate_query(
        test_case,
        index
    )

    evaluation_results.append(
        result
    )

    # Small delay to reduce API pressure
    # between Gemini calls.
    time.sleep(1)


# ================================================================
# 9. CREATE DATAFRAME
# ================================================================

evaluation_df = pd.DataFrame(
    evaluation_results
)


# ================================================================
# 10. CALCULATE OVERALL METRICS
# ================================================================

total_queries = len(
    evaluation_df
)

queries_with_candidates = int(
    (
        evaluation_df[
            "retrieved_candidates"
        ] > 0
    ).sum()
)

total_candidates = int(
    evaluation_df[
        "retrieved_candidates"
    ].sum()
)

total_region_valid = int(
    evaluation_df[
        "region_valid_candidates"
    ].sum()
)

total_season_valid = int(
    evaluation_df[
        "season_valid_candidates"
    ].sum()
)

total_maturity_valid = int(
    evaluation_df[
        "maturity_valid_candidates"
    ].sum()
)

total_possible_constraint_checks = (
    total_candidates * 3
)

total_passed_constraint_checks = (
    total_region_valid
    + total_season_valid
    + total_maturity_valid
)

if total_possible_constraint_checks > 0:

    overall_metadata_accuracy = (
        total_passed_constraint_checks
        /
        total_possible_constraint_checks
    )

else:

    overall_metadata_accuracy = 0.0


gemini_success_rate = (
    evaluation_df[
        "gemini_success"
    ].mean()
)

structure_valid_rate = (
    evaluation_df[
        "recommendation_structure_valid"
    ].mean()
)

grounding_rate = (
    evaluation_df[
        "recommendations_grounded"
    ].mean()
)

rank_valid_rate = (
    evaluation_df[
        "rank_valid"
    ].mean()
)

maturity_valid_rate = (
    evaluation_df[
        "recommendation_maturity_valid"
    ].mean()
)

irrigation_handling_rate = (
    evaluation_df[
        "irrigation_handling_valid"
    ].mean()
)

pipeline_success_rate = (
    evaluation_df[
        "pipeline_success"
    ].mean()
)

average_candidates = (
    evaluation_df[
        "retrieved_candidates"
    ].mean()
)

average_execution_time = (
    evaluation_df[
        "execution_time_seconds"
    ].mean()
)


# ================================================================
# 11. OVERALL REPORT
# ================================================================

print()
print()
print("=" * 80)
print("FINAL EVALUATION REPORT")
print("=" * 80)

print()

print(
    f"Total test queries                 : "
    f"{total_queries}"
)

print(
    f"Queries with retrieved candidates  : "
    f"{queries_with_candidates}"
)

print(
    f"Total retrieved candidates         : "
    f"{total_candidates}"
)

print(
    f"Average candidates per query       : "
    f"{average_candidates:.2f}"
)

print()

print("-" * 80)
print("METADATA FILTERING")
print("-" * 80)

print(
    f"Region matching accuracy           : "
    f"{(
        total_region_valid /
        total_candidates
        if total_candidates else 0
    ):.2%}"
)

print(
    f"Season matching accuracy           : "
    f"{(
        total_season_valid /
        total_candidates
        if total_candidates else 0
    ):.2%}"
)

print(
    f"Maturity constraint accuracy       : "
    f"{(
        total_maturity_valid /
        total_candidates
        if total_candidates else 0
    ):.2%}"
)

print(
    f"Overall metadata constraint acc.   : "
    f"{overall_metadata_accuracy:.2%}"
)

print()

print("-" * 80)
print("GEMINI RECOMMENDATION")
print("-" * 80)

print(
    f"Gemini success rate                 : "
    f"{gemini_success_rate:.2%}"
)

print(
    f"Valid JSON structure rate           : "
    f"{structure_valid_rate:.2%}"
)

print(
    f"Candidate grounding rate            : "
    f"{grounding_rate:.2%}"
)

print(
    f"Valid ranking rate                  : "
    f"{rank_valid_rate:.2%}"
)

print(
    f"Valid maturity reporting rate       : "
    f"{maturity_valid_rate:.2%}"
)

print(
    f"Valid irrigation handling rate      : "
    f"{irrigation_handling_rate:.2%}"
)

print()

print("-" * 80)
print("END-TO-END PIPELINE")
print("-" * 80)

print(
    f"Complete pipeline success rate      : "
    f"{pipeline_success_rate:.2%}"
)

print(
    f"Average execution time/query        : "
    f"{average_execution_time:.2f} seconds"
)

print()


# ================================================================
# 12. QUERY-LEVEL RESULTS TABLE
# ================================================================

display_columns = [
    "query_id",
    "retrieved_candidates",
    "metadata_constraint_accuracy",
    "gemini_success",
    "recommendations_grounded",
    "rank_valid",
    "recommendation_maturity_valid",
    "irrigation_handling_valid",
    "pipeline_success",
    "execution_time_seconds"
]

print("=" * 80)
print("QUERY-LEVEL RESULTS")
print("=" * 80)

display(
    evaluation_df[
        display_columns
    ]
)


# ================================================================
# 13. SAVE CSV REPORT
# ================================================================

timestamp = datetime.now().strftime(
    "%Y%m%d_%H%M%S"
)

csv_path = (
    PROJECT_ROOT
    / "evaluation_results_20_queries.csv"
)

evaluation_df.to_csv(
    csv_path,
    index=False
)


# ================================================================
# 14. SAVE JSON REPORT
# ================================================================

report = {

    "evaluation_title":
        "Maize Seed RAG Pipeline Evaluation",

    "evaluation_date":
        datetime.now().isoformat(),

    "number_of_test_queries":
        total_queries,

    "total_documents":
        len(documents),

    "total_retrieved_candidates":
        total_candidates,

    "average_candidates_per_query":
        round(
            average_candidates,
            4
        ),

    "metadata_evaluation": {

        "region_matching_accuracy":
            round(
                (
                    total_region_valid /
                    total_candidates
                    if total_candidates
                    else 0
                ),
                4
            ),

        "season_matching_accuracy":
            round(
                (
                    total_season_valid /
                    total_candidates
                    if total_candidates
                    else 0
                ),
                4
            ),

        "maturity_constraint_accuracy":
            round(
                (
                    total_maturity_valid /
                    total_candidates
                    if total_candidates
                    else 0
                ),
                4
            ),

        "overall_metadata_constraint_accuracy":
            round(
                overall_metadata_accuracy,
                4
            )
    },

    "gemini_evaluation": {

        "gemini_success_rate":
            round(
                gemini_success_rate,
                4
            ),

        "valid_structure_rate":
            round(
                structure_valid_rate,
                4
            ),

        "candidate_grounding_rate":
            round(
                grounding_rate,
                4
            ),

        "valid_ranking_rate":
            round(
                rank_valid_rate,
                4
            ),

        "maturity_reporting_rate":
            round(
                maturity_valid_rate,
                4
            ),

        "irrigation_handling_rate":
            round(
                irrigation_handling_rate,
                4
            )
    },

    "end_to_end": {

        "pipeline_success_rate":
            round(
                pipeline_success_rate,
                4
            ),

        "average_execution_time_seconds":
            round(
                average_execution_time,
                4
            )
    },

    "query_results":
        evaluation_results
}


json_path = (
    PROJECT_ROOT
    / "evaluation_report_20_queries.json"
)

with open(
    json_path,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        report,
        file,
        indent=4,
        ensure_ascii=False
    )


# ================================================================
# 15. PAPER-READY SUMMARY
# ================================================================

print()
print()
print("=" * 80)
print("PAPER-READY EVALUATION SUMMARY")
print("=" * 80)

print()

print(
    f"""
The proposed maize seed Retrieval-Augmented Generation (RAG)
pipeline was evaluated using {total_queries} farmer queries covering
different combinations of geographic region, cropping season,
irrigation preference, and maturity requirements.

The metadata filtering stage retrieved an average of
{average_candidates:.2f} candidates per query.

The overall metadata constraint accuracy was
{overall_metadata_accuracy:.2%}.

The Gemini recommendation component achieved a successful response
rate of {gemini_success_rate:.2%}.

The recommendation grounding rate was
{grounding_rate:.2%}, indicating the proportion of evaluated
queries for which recommended varieties originated from the
retrieved candidate set.

The valid recommendation structure rate was
{structure_valid_rate:.2%}.

The maturity information reporting validity was
{maturity_valid_rate:.2%}, while irrigation information handling
validity was {irrigation_handling_rate:.2%}.

The complete end-to-end pipeline achieved a success rate of
{pipeline_success_rate:.2%} across the 20 evaluation queries.

The average execution time was
{average_execution_time:.2f} seconds per query.
"""
)


# ================================================================
# 16. FILE LOCATIONS
# ================================================================

print()
print("=" * 80)
print("REPORT FILES")
print("=" * 80)

print()
print("CSV:")
print(csv_path)

print()
print("JSON:")
print(json_path)

print()
print("=" * 80)
print("EVALUATION COMPLETE")
print("=" * 80)

MAIZE SEED RAG - PIPELINE EVALUATION

Project root:
c:\Users\gangi\OneDrive\Desktop\Capstone Project\maize-seed-rag



Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


Total documents loaded: 302


STARTING 20-QUERY EVALUATION


EVALUATING QUERY 1/20

I am a farmer from Andhra Pradesh. I want to grow maize during Kharif season under rainfed conditions and I need a variety that matures within 100 days.

----------------------------------------------------------------------
Trying Gemini model: gemini-3.8-flash
----------------------------------------------------------------------
SERVER ERROR: gemini-3.8-flash
503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
Retrying...
SERVER ERROR: gemini-3.8-flash
503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
Moving to next model...

----------------------------------------------------------------------
Trying Gemini model: g

,query_id,retrieved_candidates,metadata_constraint_accuracy,gemini_success,recommendations_grounded,rank_valid,recommendation_maturity_valid,irrigation_handling_valid,pipeline_success,execution_time_seconds
0,Q01,6,1.0,True,True,True,True,True,True,26.05
1,Q02,14,1.0,True,True,True,True,True,True,118.29
2,Q03,5,1.0,True,True,True,True,True,True,44.12
3,Q04,4,1.0,True,True,True,True,True,True,113.41
4,Q05,7,1.0,True,True,True,True,True,True,50.32
5,Q06,13,1.0,True,True,True,True,True,True,44.55
6,Q07,12,1.0,True,True,True,True,True,True,71.00
7,Q08,13,1.0,True,True,True,True,True,True,36.57
8,Q09,13,1.0,True,True,True,True,True,True,50.69
9,Q10,3,1.0,True,True,True,True,True,True,33.81




PAPER-READY EVALUATION SUMMARY


The proposed maize seed Retrieval-Augmented Generation (RAG)
pipeline was evaluated using 20 farmer queries covering
different combinations of geographic region, cropping season,
irrigation preference, and maturity requirements.

The metadata filtering stage retrieved an average of
6.95 candidates per query.

The overall metadata constraint accuracy was
100.00%.

The Gemini recommendation component achieved a successful response
rate of 100.00%.

The recommendation grounding rate was
100.00%, indicating the proportion of evaluated
queries for which recommended varieties originated from the
retrieved candidate set.

The valid recommendation structure rate was
100.00%.

The maturity information reporting validity was
100.00%, while irrigation information handling
validity was 100.00%.

The complete end-to-end pipeline achieved a success rate of
100.00% across the 20 evaluation queries.

The average execution time was
34.26 seconds per query.


REPORT FI